In [25]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [26]:
from pathlib import Path


def find_repository_root(start=Path.cwd()):
    """Find the repository root from Jupyter's current working directory."""
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from inside "
        "NYC_Healthcare_Accessibility."
    )


REPO_ROOT = find_repository_root()
INPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_inputs"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "intermediate" / "ewm_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("Libraries loaded.")

Libraries loaded.


In [27]:

df = pd.read_csv(INPUT_DIR / "Queens! - QUEENS_INDICATORS_6.csv")
df.head(342)

,Unnamed: 0,from_id,total_distance,walking_distance,boardings,transfers,travel_time_total,fare,walking_time,wait_time_total
0,1,o_360810001011,5683.761896,1089.009,2,1,30.433333,3.0,18.433333,8.200000
1,2,o_360810001012,5417.824925,1470.173,2,1,34.800000,3.0,24.800000,8.566667
2,3,o_360810001021,5589.756896,995.004,2,1,28.866667,3.0,16.866667,9.766667
3,4,o_360810001022,5732.928896,1138.176,2,1,31.283333,3.0,19.283333,7.350000
4,5,o_360810001031,4688.543241,1281.593,2,1,30.633333,3.0,21.633333,8.000000
...,...,...,...,...,...,...,...,...,...,...
337,338,o_360810184022,4066.420031,3628.154,1,0,63.966667,3.0,61.433333,3.866667
338,339,o_360810185011,5309.967284,973.166,2,1,27.983333,3.0,16.483333,10.633333
339,340,o_360810185012,5428.461284,1091.660,2,1,29.966667,3.0,18.466667,8.650000
340,341,o_360810185013,5756.334416,942.823,2,1,28.516667,3.0,16.016667,10.100000


In [28]:
print(df.shape)
print(df.columns.tolist())

(1752, 10)
['Unnamed: 0', 'from_id', 'total_distance', 'walking_distance', 'boardings', 'transfers', 'travel_time_total', 'fare', 'walking_time', 'wait_time_total']


In [29]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

df["GEOID_TEXT"] = df["from_id"].astype(str).str.replace("o_", "", regex=False)

df[["from_id", "GEOID_TEXT"]].head(343)

,from_id,GEOID_TEXT
0,o_360810001011,360810001011
1,o_360810001012,360810001012
2,o_360810001021,360810001021
3,o_360810001022,360810001022
4,o_360810001031,360810001031
...,...,...
338,o_360810185011,360810185011
339,o_360810185012,360810185012
340,o_360810185013,360810185013
341,o_360810185021,360810185021


In [30]:
benefit_cols = []
cost_cols = ["total_distance",

    "walking_distance",

    "transfers",

    "travel_time_total",

    "walking_time",

    "wait_time_total", "fare" ]

criteria_cols = benefit_cols + cost_cols

print("Benefit indicators, higher is better:")
print(benefit_cols)

print("\nCost indicators, lower is better:")
print(cost_cols)

print("\nAll criteria:")
print(criteria_cols)

Benefit indicators, higher is better:
[]

Cost indicators, lower is better:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total', 'fare']

All criteria:
['total_distance', 'walking_distance', 'transfers', 'travel_time_total', 'walking_time', 'wait_time_total', 'fare']


In [32]:
min_max_table = pd.DataFrame({
    "min": df[criteria_cols].min(),
    "max": df[criteria_cols].max()
})

print("Min and max for each indicator:")
display(min_max_table)

Min and max for each indicator:


,min,max
total_distance,773.060609,26862.592080
walking_distance,255.648000,5403.717000
transfers,0.000000,2.000000
travel_time_total,6.866667,111.783333
walking_time,4.350000,90.983333
wait_time_total,1.016667,35.016667
fare,3.000000,9.250000


In [33]:
# All of our current indicators are cost indicators
# Lower = better, so we use: (max - value) / (max - min)

normalized = pd.DataFrame(index=df.index)

for col in criteria_cols:
    min_val = df[col].min()
    max_val = df[col].max()
    
    normalized[col] = (max_val - df[col]) / (max_val - min_val)

print("Normalized values:")
display(normalized.head())

Normalized values:


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,fare
0,0.811775,0.838122,0.5,0.775377,0.837437,0.788725,1.0
1,0.821968,0.764081,0.5,0.733757,0.763948,0.777941,1.0
2,0.815378,0.856382,0.5,0.790310,0.855521,0.742647,1.0
3,0.809890,0.828571,0.5,0.767276,0.827626,0.813725,1.0
4,0.849921,0.800713,0.5,0.773471,0.800500,0.794608,1.0


In [34]:
r_column_sums = normalized[criteria_cols].sum()

print("Step 2 preparation: Sum of each standardized column")
print("These sums go in the denominator for p_ij.")
display(r_column_sums)

Step 2 preparation: Sum of each standardized column
These sums go in the denominator for p_ij.


total_distance       1485.532714
walking_distance     1271.440275
transfers            1398.500000
travel_time_total    1231.396187
walking_time         1268.695267
wait_time_total      1511.306373
fare                 1690.480000
dtype: float64

In [35]:
P = normalized[criteria_cols] / r_column_sums

print("Step 2: Probability matrix p_ij")
print("Each standardized value is divided by its column total.")
display(P.head())

Step 2: Probability matrix p_ij
Each standardized value is divided by its column total.


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,fare
0,0.000546,0.000659,0.000358,0.000630,0.000660,0.000522,0.000592
1,0.000553,0.000601,0.000358,0.000596,0.000602,0.000515,0.000592
2,0.000549,0.000674,0.000358,0.000642,0.000674,0.000491,0.000592
3,0.000545,0.000652,0.000358,0.000623,0.000652,0.000538,0.000592
4,0.000572,0.000630,0.000358,0.000628,0.000631,0.000526,0.000592


In [36]:
print("values should add up to one for each column, since they are probabilities.")
display(P.sum())

values should add up to one for each column, since they are probabilities.


total_distance       1.0
walking_distance     1.0
transfers            1.0
travel_time_total    1.0
walking_time         1.0
wait_time_total      1.0
fare                 1.0
dtype: float64

In [37]:
P_safe = P.replace(0, 1e-12)

print("Step 3 preparation: Replace 0 values so ln(0) does not break")
display(P_safe.head())

Step 3 preparation: Replace 0 values so ln(0) does not break


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,fare
0,0.000546,0.000659,0.000358,0.000630,0.000660,0.000522,0.000592
1,0.000553,0.000601,0.000358,0.000596,0.000602,0.000515,0.000592
2,0.000549,0.000674,0.000358,0.000642,0.000674,0.000491,0.000592
3,0.000545,0.000652,0.000358,0.000623,0.000652,0.000538,0.000592
4,0.000572,0.000630,0.000358,0.000628,0.000631,0.000526,0.000592


In [38]:
ln_P = np.log(P_safe)

print("Step 3: Natural log of p_ij")
display(ln_P.head())

Step 3: Natural log of p_ij


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,fare
0,-7.512061,-7.324498,-7.936303,-7.370309,-7.323153,-7.558067,-7.432768
1,-7.499582,-7.416987,-7.936303,-7.425481,-7.415000,-7.571834,-7.432768
2,-7.507632,-7.302944,-7.936303,-7.351234,-7.301789,-7.618264,-7.432768
3,-7.514385,-7.335958,-7.936303,-7.380813,-7.334938,-7.526862,-7.432768
4,-7.466140,-7.370159,-7.936303,-7.372771,-7.368263,-7.550636,-7.432768


In [39]:
P_ln_P = P_safe * ln_P

print("Step 4: p_ij times ln(p_ij)")
display(P_ln_P.head(8))

Step 4: p_ij times ln(p_ij)


,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,fare
0,-0.004105,-0.004828,-0.002837,-0.004641,-0.004834,-0.003944,-0.004397
1,-0.004150,-0.004457,-0.002837,-0.004425,-0.004465,-0.003898,-0.004397
2,-0.004121,-0.004919,-0.002837,-0.004718,-0.004924,-0.003744,-0.004397
3,-0.004097,-0.004781,-0.002837,-0.004599,-0.004785,-0.004053,-0.004397
4,-0.004272,-0.004641,-0.002837,-0.004631,-0.004649,-0.003970,-0.004397
5,-0.004217,-0.004455,-0.002837,-0.004473,-0.004463,-0.003774,-0.004397
6,-0.003154,-0.004333,-0.002837,-0.003606,-0.004338,-0.004135,-0.004397
7,-0.003197,-0.004571,-0.002837,-0.003813,-0.004574,-0.003619,-0.004397


In [40]:
p_ln_p_sums = P_ln_P.sum(axis=0)

print("Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator")
display(p_ln_p_sums)

Step 4 preparation: Sum of p_ij * ln(p_ij) for each indicator


total_distance      -7.454282
walking_distance    -7.441017
transfers           -7.397546
travel_time_total   -7.437574
walking_time        -7.440604
wait_time_total     -7.458877
fare                -7.448607
dtype: float64

In [41]:
# Step 5: Calculate entropy for each indicator

n = len(normalized)
k = 1 / np.log(n)

entropy = -k * p_ln_p_sums

print("Number of rows:", n)
print("k value:", k)
print("Step 5: Entropy for each indicator")
display(entropy)

Number of rows: 1752
k value: 0.13389545732167485
Step 5: Entropy for each indicator


total_distance       0.998094
walking_distance     0.996318
transfers            0.990498
travel_time_total    0.995857
walking_time         0.996263
wait_time_total      0.998710
fare                 0.997335
dtype: float64

In [42]:
df["fare"].value_counts()

fare
3.00    1654
6.25      76
9.25      22
Name: count, dtype: int64

In [43]:
# Step 6: Diversity
# Diversity tells us how much useful variation each indicator has

diversity = 1 - entropy

print("Step 6: Diversity for each indicator")
display(diversity)

Step 6: Diversity for each indicator


total_distance       0.001906
walking_distance     0.003682
transfers            0.009502
travel_time_total    0.004143
walking_time         0.003737
wait_time_total      0.001290
fare                 0.002665
dtype: float64

In [44]:
# Step 7: Calculate entropy weights
# Weight = diversity of one indicator / total diversity of all indicators

weights = diversity / diversity.sum()

print("Step 7: Entropy weights for each indicator")
display(weights)

Step 7: Entropy weights for each indicator


total_distance       0.070773
walking_distance     0.136741
transfers            0.352920
travel_time_total    0.153859
walking_time         0.138791
wait_time_total      0.047921
fare                 0.098996
dtype: float64

In [45]:
weights_table = pd.DataFrame({
    "entropy": entropy,
    "diversity": diversity,
    "weight": weights
})

print("Final entropy weight table:")
display(weights_table)

Final entropy weight table:


,entropy,diversity,weight
total_distance,0.998094,0.001906,0.070773
walking_distance,0.996318,0.003682,0.136741
transfers,0.990498,0.009502,0.352920
travel_time_total,0.995857,0.004143,0.153859
walking_time,0.996263,0.003737,0.138791
wait_time_total,0.998710,0.001290,0.047921
fare,0.997335,0.002665,0.098996


In [46]:
display(weights_table.sort_values(by="weight", ascending=False))

,entropy,diversity,weight
transfers,0.990498,0.009502,0.352920
travel_time_total,0.995857,0.004143,0.153859
walking_time,0.996263,0.003737,0.138791
walking_distance,0.996318,0.003682,0.136741
fare,0.997335,0.002665,0.098996
total_distance,0.998094,0.001906,0.070773
wait_time_total,0.998710,0.001290,0.047921


In [47]:
# Step 8: Calculate final EWM accessibility score
# Formula: score for each block group = sum(normalized value * indicator weight)

df["ewm_accessibility_score"] = (normalized[criteria_cols] * weights).sum(axis=1)

print("Step 8: Final EWM accessibility score")
display(df[["from_id", "GEOID_TEXT", "ewm_accessibility_score"]].head(30))

Step 8: Final EWM accessibility score


,from_id,GEOID_TEXT,ewm_accessibility_score
0,o_360810001011,360810001011,0.720837
1,o_360810001012,360810001012,0.694314
2,o_360810001021,360810001021,0.726188
3,o_360810001022,360810001022,0.717987
4,o_360810001031,360810001031,0.713283
5,o_360810001041,360810001041,0.695361
6,o_360810002001,360810002001,0.650673
7,o_360810002002,360810002002,0.664461
8,o_360810004001,360810004001,0.648150
9,o_360810004002,360810004002,0.658084


In [48]:
print("Score summary:")
display(df["ewm_accessibility_score"].describe())

Score summary:


count    1752.000000
mean        0.786455
std         0.131517
min         0.279479
25%         0.692809
50%         0.808444
75%         0.897335
max         0.996391
Name: ewm_accessibility_score, dtype: float64

In [50]:
final_results = df[
    ["GEOID_TEXT", "from_id"] + criteria_cols + ["ewm_accessibility_score"]
].copy()

display(final_results.head(20))

,GEOID_TEXT,from_id,total_distance,walking_distance,transfers,travel_time_total,walking_time,wait_time_total,fare,ewm_accessibility_score
0,360810001011,o_360810001011,5683.761896,1089.009,1,30.433333,18.433333,8.200000,3.0,0.720837
1,360810001012,o_360810001012,5417.824925,1470.173,1,34.800000,24.800000,8.566667,3.0,0.694314
2,360810001021,o_360810001021,5589.756896,995.004,1,28.866667,16.866667,9.766667,3.0,0.726188
3,360810001022,o_360810001022,5732.928896,1138.176,1,31.283333,19.283333,7.350000,3.0,0.717987
4,360810001031,o_360810001031,4688.543241,1281.593,1,30.633333,21.633333,8.000000,3.0,0.713283
5,360810001041,o_360810001041,5012.883815,1472.179,1,33.833333,24.833333,9.533333,3.0,0.695361
6,360810002001,o_360810002001,11219.266820,1596.471,1,50.966667,26.966667,6.700000,3.0,0.650673
7,360810002002,o_360810002002,10976.861820,1354.066,1,46.933333,22.933333,10.733333,3.0,0.664461
8,360810004001,o_360810004001,10678.559480,1670.801,1,50.733333,28.233333,6.933333,3.0,0.648150
9,360810004002,o_360810004002,11088.905820,1466.110,1,48.800000,24.800000,8.866667,3.0,0.658084


In [51]:
final_results.to_csv(OUTPUT_DIR / "queens_ewm_results.csv", index=False)
weights_table.to_csv(OUTPUT_DIR / "queens_ewm_weights.csv", index=True)

print("Saved queens_ewm_results.csv")
print("Saved queens_ewm_weights.csv")

Saved queens_ewm_results.csv
Saved queens_ewm_weights.csv
